# Module 6 Assignment: Give Your Persona Agent More Tools

In Module 5 you built a Persona Agent with one tool. This assignment makes the same jump Lab 5
made on top of Lab 4: your agent goes from one tool to three, and `chat()` has to learn to route
between them instead of assuming.

You'll reuse your own Module 5 code at almost every step, plus the routing pattern from Lab 5.
This assignment is about **extending** what you already built, not writing it from scratch.

**What you'll build:**
1. Your persona, notes, and first tool, carried over from Module 5
2. A second tool, backed by a free, no-key external API
3. A third tool of your own design
4. JSON descriptions for all three tools
5. An updated system prompt that tells the model when to use each tool
6. A `call_tool(name, arguments)` dispatcher, exactly like Lab 5's
7. A full agent loop that routes across all three tools

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you can find where Lab 5 made this same change to Lab 4, you can make it here too.

## 0. Install the packages (run once)

Same as Module 5, plus one new package: `requests`, for calling your new tool's API.

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv requests

Resolved 63 packages in 3.25s
 Downloaded hf-xet
 Downloaded openai
Prepared 4 packages in 22.30s
Uninstalled 4 packages in 4.29s
Installed 4 packages in 7.90s
 - hf-xet==1.6.0
 + hf-xet==1.7.0
 - openai==3.24.0
 + openai==3.26.0
 - opentelemetry-api==1.45.0
 + opentelemetry-api==1.45.1
 - typer==0.27.2
 + typer==0.27.3


## 1. Imports

**TODO:** Copy the imports from your Module 5 assignment, plus `requests` for calling an external
API. Add any other standard library module your second or third tool needs (for example `html`,
if the API you pick returns HTML-escaped text the way Open Trivia Database does).

In [2]:
from dotenv import load_dotenv
import os
import sys
import json
import html

import requests
from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr



c:\Users\Danie\Downloads\Assignment 6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

Identical to every previous lab and assignment. Copy this cell directly from Module 5.

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment - fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins 5624532c


## 3. Recreate your persona, notes, and first tool from Module 5

Bring forward everything you already built: your `persona`, your `notes/notes.pdf` loading code,
your one tool function from Module 5, and that tool's JSON description. Nothing here needs to
change, this is your starting point.

**TODO:** Paste in, from your Module 5 assignment:
- Your `persona` variable
- Your `PdfReader("notes/notes.pdf")` loading code that builds `notes`
- Your Module 5 tool function (rename it if you like, but keep track of its name)
- That tool's JSON schema dictionary

**Hint:** copy your `notes/notes.pdf` file into this assignment's folder too, in a `notes`
subfolder, exactly like Lab 5 did with Lab 4's notes file.

In [4]:
persona = (
    "Imperial intelligence has asked that a test on local insurgent knowledge must be used in order to succed in the Empire's goal to control the galaxy."
)

reader = PdfReader("Imperialnotes/ImperialNotes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text


def log_missed_question_tool(question):
    print(f"Tool called to log a missed question: {question}")
    with open("missed_questions.txt", "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Missed question logged"

print(persona)
print(notes[:300], "...")

Imperial intelligence has asked that a test on local insurgent knowledge must be used in order to succed in the Empire's goal to control the galaxy.
IMPERIAL INTELLIGENCE 
RESTRICTED FIELD NOTES 
SUBJECT: Imperial Worlds with Suspected Rebel Insurgent Activity 
Classification: Imperial Eyes Only 
Prepared for: Imperial Security Bureau / Sector Command 
Status: Active Intelligence File 
Primary Concern: Rebel Alliance recruitment, smuggling, espi ...


In [5]:
my_tool_json = {
    "name": "my_tool",  "persona"
    "description": "TODO: describe when the model should call this tool",
    "parameters": {
        "type": "object",
        "properties": {
            "value": {"type": "string", "description": "TODO: describe this argument"}
        },
        "required": ["value"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": my_tool_json}]


## 4. New tool: something backed by a free API

Design a **second** tool for your Persona Agent, this one calling a free, no-key external API,
the same way Lab 5's `fetch_bonus_question_tool` calls the Open Trivia Database.

**Ideas, pick whichever fits your topic, or find your own:**
- A trivia-friendly agent could call the Open Trivia Database for a bonus question
- A weather-aware topic could call [Open-Meteo](https://open-meteo.com), a free weather API that
  needs no key
- A language-learning agent could call the [Free Dictionary API](https://dictionaryapi.dev) to
  define a word, though it is sometimes slow or briefly unavailable, so handle that the same way
  the next tip describes
- Any other free, no-key API relevant to your topic works too, just check it does not require
  signup or a paid key

**TODO:** Write your tool function below. It should make one `requests.get(...)` call inside a
`try`/`except` that catches `requests.exceptions.RequestException`, so a slow or unavailable API
returns a short, friendly string instead of crashing your notebook, exactly like Lab 5's tools
do.

In [6]:
OPENTDB_CATEGORIES = {"Planet Knowledge": 9, "Space Stations": 18}

def fetch_bonus_question_tool(category=None):
    print(f"Tool called to fetch a bonus trivia question (category={category})")
    params = {"amount": 1, "type": "multiple"}
    if category in OPENTDB_CATEGORIES:
        params["category"] = OPENTDB_CATEGORIES[category]

    try:
        response = requests.get("https://opentdb.com/api.php", params=params, timeout=20)
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"fetch_bonus_question_tool network error: {e}")
        return "The trivia API is temporarily unavailable - just skip the bonus question and continue the quiz."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a bonus question right now - just skip it and continue the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])
    return f"Bonus question: {question} (correct answer: {answer})"

In [7]:
fetch_bonus_question_tool("Space Stations")

Tool called to fetch a bonus trivia question (category=Space Stations)


'Bonus question: What does the acronym CDN stand for in terms of networking? (correct answer: Content Delivery Network)'

## 5. New tool: one more tool of your own design

Design a **third** tool. It can be another local tool in the spirit of Module 5 (a running score
tracker, a bookmarking tool for topics that need review, or an idea of your own), or a second
external API call if you would rather. Either is fine, as long as it does something the first two
tools do not already cover.

**TODO:** Write your third tool function below.

In [8]:
def check_progress_tool():
    print("Tool called to check quiz progress")
    try:
        with open("missed_questions.txt", "r", encoding="utf-8") as f:
            missed = [line for line in f if line.strip()]
    except FileNotFoundError:
        missed = []

    if not missed:
        return "No questions have been missed yet - a clean sheet so far!"
    return f"So far, {len(missed)} question(s) have been logged as missed."

In [9]:
check_progress_tool() # quick manual test of your third tool

Tool called to check quiz progress


'No questions have been missed yet - a clean sheet so far!'

## 6. Describe all three tools to the model with JSON

Same idea as Module 5 and Lab 5, just three schemas now. Remember an argument can be optional (an
empty `"required"` list), exactly like Lab 5's `fetch_bonus_question_tool`, if that fits your
tool.

In [10]:
log_missed_question_tool_json = {
    "name": "log_missed_question_tool",
    "description": "Use this tool to record a quiz question the student got wrong or didn't know, so they can review it later",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The exact quiz question the student missed"}
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

fetch_bonus_question_tool_json = {
    "name": "fetch_bonus_question_tool",
    "description": "Use this tool to fetch a fun bonus trivia question from an external trivia database, when the student asks for a bonus question or a break from the study notes",
    "parameters": {
        "type": "object",
        "properties": {
            "category": {
                "type": "string",
                "enum": ["general_knowledge", "science_computers"],
                "description": "Which trivia category to pull a bonus question from",
            }
        },
        "required": [],
        "additionalProperties": False,
    },
}

check_progress_tool_json = {
    "name": "check_progress_tool",
    "description": "Use this tool when the student asks how they are doing, how many questions they've missed so far, or asks for a progress check",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": [],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": log_missed_question_tool_json},
    {"type": "function", "function": fetch_bonus_question_tool_json},
    {"type": "function", "function": check_progress_tool_json},
]


## 7. Update your system prompt for three tools

**TODO:** Build your `system_prompt` f-string the same way you did in Module 5, still using
`persona` and `notes`, but now with three rules telling the model when to reach for each tool,
the same way Lab 5's system prompt added two new sentences to Lab 4's.

In [11]:
system_prompt = f"""

# Your role

This is {persona}

You quiz a Stormtrooper on the notes below.

# Context

Here are the study notes you are quizzing the Trooper on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the Trooper's answer
before asking the next question.

After the Trooper answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

If the Trooper's answer was wrong, or they say "I don't know", you must record it as a missed
question (using the tool available to you) before moving on to the next question.

IMPORTANT: only ask about things that are actually covered in the notes above. If the Trooper asks
about something outside the notes, say that it isn't covered in this study set.

BONUS: If there's any bonus questions added to the list, then it will save it in a different file than the other questions.
"""


display(Markdown(system_prompt))



# Your role

This is Imperial intelligence has asked that a test on local insurgent knowledge must be used in order to succed in the Empire's goal to control the galaxy.

You quiz a Stormtrooper on the notes below.

# Context

Here are the study notes you are quizzing the Trooper on:

IMPERIAL INTELLIGENCE 
RESTRICTED FIELD NOTES 
SUBJECT: Imperial Worlds with Suspected Rebel Insurgent Activity 
Classification: Imperial Eyes Only 
Prepared for: Imperial Security Bureau / Sector Command 
Status: Active Intelligence File 
Primary Concern: Rebel Alliance recruitment, smuggling, espionage, sabotage, and 
concealed military operations. 
 
1. LOTHAL 
Sector: Lothal Sector 
Primary Population: Human and Lothalite civilian population 
Imperial Importance: Agricultural production, manufacturing, Imperial military 
infrastructure 
Senator / Political Representative 
Senator: Gall Trayvis 
Status: Publicly presented himself as a critic of Imperial rule, though his activities and 
loyalties were later called into question. 
Known Ships 
• VCX-100 light freighter Ghost 
• TIE fighters 
• Imperial cargo shuttles 
• Various civilian freighters and transports 
• Rebel transport vessels operating under civilian cover 
Suspected Contraband 
• Stolen Imperial weapons 
• Blaster rifles 
• Explosives 
• Imperial communications equipment • Stolen military supplies 
• Recovered TIE components 
• Intelligence data taken from Imperial facilities 
Suspected Hidden Outposts 
• Abandoned agricultural buildings 
• Old transportation depots 
• Remote farming settlements 
• Hidden caves and wilderness locations outside major Imperial installations 
• Temporary safe houses around Capital City 
Known / Suspected Dissidents 
• Ezra Bridger — Local insurgent who became a member of the Ghost crew. 
• Hera Syndulla — Rebel operative and commander of the Ghost. 
• Kanan Jarrus — Jedi and Rebel operative. 
• Sabine Wren — Weapons specialist and former Imperial Academy cadet. 
• Zeb Orrelios — Rebel operative frequently involved in military actions. 
• Garazeb Orrelios — Known enemy combatant. 
Imperial Assessment 
Lothal should be considered a significant source of Rebel recruitment and material 
support. Civilian resistance is particularly difficult to distinguish from ordinary 
dissatisfaction with Imperial policies. 
 
2. ALDERAAN 
Sector: Alderaan Sector 
Imperial Importance: Political influence, diplomatic connections 
Senator 
Senator: Bail Organa 
Status: Former senator and suspected major supporter of anti-Imperial activity. Known Ships 
• Alderaanian diplomatic vessels 
• CR90 corvettes 
• Civilian transports 
• Diplomatic shuttles 
• Rebel-operated transports connected to Alderaanian personnel 
Suspected Contraband 
• Rebel intelligence 
• Weapons 
• Communications equipment 
• Forged travel documentation 
• Personnel records 
• Secret diplomatic correspondence 
Suspected Hidden Outposts 
Prior to the destruction of Alderaan: 
• Private estates 
• Diplomatic facilities 
• Hidden communications rooms 
• Civilian transport hubs 
Following the destruction of Alderaan, surviving Alderaanian communities became 
scattered throughout the galaxy. 
Known Dissidents 
• Bail Organa 
• Breha Organa 
• Leia Organa 
• Alderaanian diplomatic personnel and survivors Imperial Assessment 
Alderaan represented a major political center of resistance before its destruction. Surviving 
Alderaanian citizens and military personnel should be monitored for connections to the 
Rebel Alliance. 
 
3. MON CALA 
Sector: Mon Calamari Sector 
Imperial Importance: Shipbuilding, naval resources 
Political Leadership 
Mon Calamari Leader: King Lee-Char 
Political Representative: Mon Mothma maintained significant political connections with 
Mon Calamari resistance networks. 
Known Ships 
• MC80 Star Cruisers 
• Mon Calamari civilian transports 
• Quarren and Mon Calamari freighters 
• Imperialized local vessels 
• Rebel fleet support vessels 
Suspected Contraband 
• Starfighter components 
• Naval weapons 
• Military-grade computers 
• Shipbuilding equipment 
• Rebel communications equipment 
• Fuel and reactor components 
Suspected Hidden Outposts 
• Underwater facilities • Remote ocean settlements 
• Shipyard storage facilities 
• Civilian cargo vessels 
• Deep-ocean communications stations 
Known Dissidents 
• Mon Mothma 
• Admiral Raddus 
• Mon Calamari and Quarren resistance fighters 
Imperial Assessment 
Mon Cala possesses unusually valuable naval capabilities. Any organized resistance 
movement operating there could potentially obtain warships rather than merely small 
arms. 
 
4. CORELLIA 
Sector: Corellian Sector 
Imperial Importance: Major shipbuilding and transportation center 
Political Representative 
Representative/Senatorial Connections: Corellian political representation has 
historically been divided between local authorities and Imperial institutions. 
Known Ships 
• Corellian Engineering Corporation freighters 
• YT-series light freighters 
• CR90 Corvettes 
• Corellian corvettes 
• Modified civilian transports 
• Imperial patrol vessels 
Suspected Contraband • Weapons 
• Hyperdrive components 
• Military-grade engines 
• Stolen Imperial spare parts 
• Starfighter components 
• Forged identification documents 
Suspected Hidden Outposts 
• Shipyard maintenance areas 
• Smuggling warehouses 
• Abandoned industrial facilities 
• Freighter repair stations 
• Remote landing fields 
Known Dissidents 
• Han Solo — Former Corellian pilot and smuggler who later became associated with 
the Rebel Alliance. 
• Garm Bel Iblis — Corellian political figure who became involved with the Rebel 
movement. 
• Various Corellian smugglers and independent pilots. 
Imperial Assessment 
Corellia's enormous civilian shipping industry makes it exceptionally difficult to identify 
Rebel vessels. A military vessel can potentially be disguised as an ordinary freighter or 
courier. 
 
5. NAL HUTTA 
Sector: Y'toub System 
Imperial Importance: Major criminal and commercial networks 
Political Authority Local Ruler: Jabba Desilijic Tiure and the Hutt leadership structure. 
Known Ships 
• Hutt transports 
• Smuggling freighters 
• Armed pleasure craft 
• Cargo ships 
• Modified civilian vessels 
Suspected Contraband 
• Spice 
• Illegal weapons 
• Stolen Imperial equipment 
• Forged identification 
• Restricted technology 
• Military supplies 
• Intelligence information 
Suspected Hidden Outposts 
• Hutt-controlled warehouses 
• Private docking facilities 
• Smuggler safe houses 
• Remote asteroid bases 
• Criminal syndicate facilities 
Known Dissidents / Anti-Imperial Actors 
The planet is less associated with organized ideological rebellion than with criminal groups 
willing to cooperate with or oppose the Empire depending upon financial interests. 
Imperial Assessment Rebel agents may exploit Hutt-controlled territory because Imperial authority is 
comparatively limited. Smugglers operating through the Hutt criminal network represent a 
significant security concern. 
 
6. KESSEL 
Sector: Kessel Sector 
Imperial Importance: Mining and resource extraction 
Imperial Facilities 
Kessel contains Imperial-controlled mining operations and prison labor facilities. 
Known Ships 
• Imperial prison transports 
• Cargo freighters 
• Mining transports 
• YT-series freighters 
• Smuggling vessels 
Suspected Contraband 
• Coaxium 
• Spice 
• Stolen mining equipment 
• Weapons 
• Prisoner identification records 
• Imperial communication equipment 
Suspected Hidden Outposts 
• Abandoned mining tunnels 
• Remote asteroid facilities 
• Smuggler stations 
• Underground storage areas Known Dissidents 
• Prisoners involved in organized resistance 
• Smugglers 
• Independent miners 
• Criminal groups opposing Imperial control 
Imperial Assessment 
The planet's extensive underground mining infrastructure creates numerous locations 
where personnel and equipment can be concealed. 
 
7. RYLOTH 
Sector: Twi'lek Sector 
Imperial Importance: Natural resources and strategic location 
Political Representative 
Political Leadership: Cham Syndulla and other local Twi'lek leaders have maintained 
resistance networks against Imperial occupation. 
Known Ships 
• Ryloth transports 
• Modified freighters 
• Rebel starfighters 
• Armed civilian vessels 
• Imperial occupation ships 
Suspected Contraband 
• Blasters 
• Explosives 
• Medical supplies 
• Communications equipment 
• Stolen Imperial weapons • Food shipments intended for resistance groups 
Suspected Hidden Outposts 
• Mountain caves 
• Remote villages 
• Underground shelters 
• Abandoned mining facilities 
• Desert landing areas 
Known Dissidents 
• Cham Syndulla 
• Hera Syndulla 
• Twi'lek resistance fighters 
Imperial Assessment 
Resistance activity is deeply rooted in local communities. Imperial attempts to eliminate 
individual cells may fail because replacement personnel and supplies can emerge from 
civilian populations. 
 
8. DANTOOINE 
Sector: Dantooine Sector 
Imperial Importance: Low strategic value, making it useful for covert activity 
Political Representative 
No major Imperial senator currently identified as a primary Rebel suspect. 
Known Ships 
• Civilian freighters 
• Light transports 
• Rebel courier vessels 
• Small patrol craft 
Suspected Contraband • Weapons 
• Medical supplies 
• Rebel communications equipment 
• Food 
• Stolen Imperial documents 
Suspected Hidden Outposts 
• Remote farms 
• Abandoned military installations 
• Caves 
• Wilderness camps 
• Temporary Rebel supply depots 
Known Dissidents 
• Local farmers 
• Former military personnel 
• Independent traders 
• Rebel sympathizers 
Imperial Assessment 
Dantooine's relatively low Imperial profile makes it a possible staging location for Rebel 
personnel attempting to disappear from Imperial surveillance. 
 
9. SULLUST 
Sector: Sullust System 
Imperial Importance: Industrial production and strategic hyperspace location 
Political Representative 
Local political leadership is heavily influenced by Imperial and corporate interests. 
Known Ships 
• SoroSuub Corporation vessels • Sullustan transports 
• Cargo freighters 
• Rebel fleet support vessels 
• Imperial patrol craft 
Suspected Contraband 
• Starship components 
• Fuel 
• Weapons 
• Navigation equipment 
• Industrial machinery 
• Rebel communications equipment 
Suspected Hidden Outposts 
• Industrial tunnels 
• Mining facilities 
• Cargo warehouses 
• Remote settlements 
• Underground maintenance areas 
Known Dissidents 
• Nien Nunb — Sullustan pilot later associated with the Rebel Alliance. 
• Various Sullustan pilots and industrial workers. 
Imperial Assessment 
The planet's industrial infrastructure provides opportunities to conceal Rebel equipment 
among legitimate cargo. 
 
10. GLEE ANSELM 
Sector: Glee Anselm Sector 
Imperial Importance: Moderate Political Situation 
The planet contains both Nautolan and Anselmi populations with varying attitudes toward 
Imperial authority. 
Known Ships 
• Civilian transports 
• Nautolan vessels 
• Fishing and cargo craft 
• Local patrol ships 
Suspected Contraband 
• Weapons 
• Communications equipment 
• Stolen Imperial supplies 
• Medical equipment 
Suspected Hidden Outposts 
• Coastal settlements 
• Underwater facilities 
• Remote islands 
• Abandoned industrial buildings 
Known Dissidents 
Local resistance activity is believed to consist primarily of small cells rather than a large 
centralized organization. 
Imperial Assessment 
Underwater and coastal geography makes conventional Imperial surveillance difficult. 
 
GENERAL INDICATORS OF REBEL ACTIVITY 
Imperial personnel should watch for the following patterns: 
Transportation • Civilian freighters making unexplained hyperspace jumps 
• Ships carrying unusually large amounts of fuel 
• Damaged vessels avoiding Imperial repair facilities 
• Repeated visits to remote moons or asteroids 
• Ships using falsified transponder codes 
Contraband 
• Military-grade weapons disguised as civilian cargo 
• Stolen Imperial uniforms 
• Restricted medical supplies 
• Encrypted communications equipment 
• Starfighter components 
• Explosives 
• Imperial data cylinders 
• Forged identification documents 
Personnel 
Individuals should receive additional scrutiny when they: 
• Publicly criticize Imperial authority 
• Refuse Imperial military contracts 
• Maintain unexplained connections with smugglers 
• Frequently travel between politically unstable worlds 
• Have military training without current Imperial employment 
• Demonstrate unusual knowledge of Imperial facilities 
 
HIGH-RISK REBEL INDICATORS 
Priority Alpha 
• Confirmed Rebel personnel • Rebel fleet vessels 
• Stolen Imperial weapons shipments 
• Hidden military installations 
• Imperial personnel supplying insurgents 
Priority Beta 
• Suspected safe houses 
• Smuggling networks 
• Anti-Imperial political organizations 
• Unregistered civilian starships 
Priority Gamma 
• Individual political dissent 
• Anti-Imperial statements 
• Unauthorized gatherings 
• Suspicious civilian travel 
 
FINAL INTELLIGENCE NOTE 
The presence of anti-Imperial sentiment should not automatically be interpreted as proof 
of Rebel Alliance membership. Many citizens may object to individual Imperial policies 
without participating in an organized insurgency. 
Particular attention should therefore be given to connections between political 
dissidents, smugglers, stolen military equipment, unidentified starships, and 
unexplained movements of personnel. 
END OF REPORT 
IMPERIAL SECURITY BUREAU 
UNAUTHORIZED DISTRIBUTION PUNISHABLE UNDER IMPERIAL LAW 
 

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the Trooper's answer
before asking the next question.

After the Trooper answers, say clearly whether they were right or wrong, and give a one-sentence
explanation either way, staying in character the whole time.

If the Trooper's answer was wrong, or they say "I don't know", you must record it as a missed
question (using the tool available to you) before moving on to the next question.

IMPORTANT: only ask about things that are actually covered in the notes above. If the Trooper asks
about something outside the notes, say that it isn't covered in this study set.

BONUS: If there's any bonus questions added to the list, then it will save it in a different file than the other questions.


## 8. The routing problem: which tool did the model actually call?

With three tools, `chat()` can no longer assume. It has to check `tool_call.function.name` and
call the matching Python function.

**TODO:** Copy the `call_tool(name, arguments)` dispatcher pattern from Lab 5, Step 9, but route
across your own three tool names instead.

In [12]:
def call_tool(name, arguments):
    if name == "log_missed_question_tool":
        return log_missed_question_tool(arguments.get("question"))
    elif name == "fetch_bonus_question_tool":
        return fetch_bonus_question_tool(arguments.get("category"))
    elif name == "check_progress_tool":
        return check_progress_tool()
    else:
        return f"Unknown tool: {name}"


## 9. Update `chat()` to route across all three tools

**TODO:** Copy the finished agent loop from your Module 5 assignment. The `while` loop and the
`for tool_call in message_obj.tool_calls` loop do not change at all, the only new line is a call
to `call_tool(...)` in place of your old single hardcoded tool call.

In [13]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)
        for tool_call in message_obj.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            result = call_tool(name, arguments)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

## 10. Try your multi-tool Persona Agent

Launch the chat interface and try to trigger all three tools in one session: something that
should hit your original Module 5 tool, something that should hit your new API-backed tool, and
something that should hit your third tool. Watch the terminal or output to see which tool fires
each time. Interrupt the cell when you're done.

In [14]:
gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Reflection questions

Answer these in a new Markdown cell below.

1. What are your three tools, and what does each one do? Why did you choose the second and third
   tools you designed?
   Tool 1: Logs missing question. Designed to log a question if you missed it.
   Tool 2: Get question from online database. Designed to call a random question from the database from an online database website. 
   Tool 3: Quiz Progress. Checks the quiz to make certain that the quiz that you are taking is on schedule progress.
   I chose these two tools for my design since Imperial Intellegence should have a bonus question outside of rebel propaganda and focus on what each planet has. With the quiz progress tool, it also shows how much a trooper has completed the quiz and if they are on time to finish it.
2. Compare this assignment to Module 5. What was exactly the same as going from one tool to more
   than one? What, if anything, surprised you?
   Nothing really surprised me and the exact same thing from going from one to more tool is just adding more to the Json code section instead of just having missed question. 
3. Walk through what `call_tool` actually does the first time your agent calls your new
   API-backed tool. What would break if you deleted the `elif` branch for that tool?
   If I deleted the elif branch for that tool, it would crashed that section of the program and refuse to run it because the elif loop would go on forever and ever. 
4. Open Lab 5's system prompt and compare it to yours. Both add rules for two new tools, using
   only plain English sentences and no code changes to the agent loop. Why does that work, in
   terms of what a system prompt actually is?
   A system prompt is basically instruction on what something is supposed to do before carrying out the steps to do it. The reason it works this way is because once a user types in a prompt into the system, the machine will read over your prompt, look at the steps it has been instructed, execute the steps, and show the final results below. 

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit, it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, for example `Module6Assignment_JaneDoe.ipynb`, and
  submit it.

You've now made the same jump Lab 5 made: one tool to several, with a real dispatcher and a loop
that already scaled to handle it.